# Defining custom markets and an associated fleet

AeroMAPS ships with a default 4-market structure (`short_range`, `medium_range`, `long_range`, `freight`), but you can replace it with **user-defined markets** declared in YAML. The market concept is the top-level grouping for traffic, energy intensity, and fleet renewal — downstream models (energy, emissions, costs) auto-adapt to whatever markets you declare.

This tutorial walks through a minimal example with **three markets**: two passenger markets (replacing the default short/medium/long split) and one freight market.

| Market id | Name | Type | RPK / RTK share 2019 | Energy share 2019 |
|-----------|------|------|:--------------------:|:-----------------:|
| `regional` | Regional | passenger | 62.3 % | 52.19 % |
| `intercontinental` | Intercontinental | passenger | 37.7 % | 32.81 % |
| `air_cargo` | Air Cargo | freight | 100 % | 15 % |

We will:
1. Write the three YAML files that describe the scenario.
2. Point a `config.yaml` at them.
3. Build the process, run it, and inspect per-market results.

## 1. The four input files

A custom scenario is fully described by **four YAML files**, all kept side-by-side under `data/`:

| File | Role |
|------|------|
| `custom_markets.yaml` | declares each market — id, traffic type, 2019 calibration, growth, COVID, efficiency... |
| `custom_fleet.yaml` | declares the fleet structure for each *passenger* market (subcategories, shares) |
| `custom_aircraft_inventory.yaml` | declares the reference + new aircraft used by the fleet |
| `config.yaml` | wires the three together and selects which AeroMAPS models to run |

Let's look at each in turn.

### 1.a — `custom_markets.yaml`

Each top-level key (other than `global` and `defaults`) declares one market. The required keys are `name`, `traffic_type` (`passenger`|`freight`), `traffic_unit` (`RPK`|`RTK`), and `inputs.initial` (2019 calibration). All other input blocks (`growth`, `covid`, `load_factor`, `measures`, `efficiency_simple`, `costs`, `reference`) are *optional* — anything you omit falls back to the values declared under `defaults`.

**Conservation constraints:**
* `rpk_share_2019` across all passenger markets must sum to 100.
* `energy_share_2019` across **all** markets (passenger + freight) must sum to 100.
* For freight markets, the per-market `rtk_share_2019` must sum to 100 across freight markets (here we have a single freight market, so it is 100).

In [ ]:
from pathlib import Path

print(Path("data/custom_markets.yaml").read_text())

### 1.b — `custom_fleet.yaml`

Declares the fleet structure for each **passenger** market (freight markets are handled through the freight efficiency model and need no fleet entry). Two top-level blocks:

* `subcategories:` — the subcategory pool. Each subcategory has a `share` within its market (subcategory shares **must sum to 100 % per market**), optional `reference_aircraft` (old + recent baseline) and a list of `aircraft:` entries.
* `categories:` — one entry per passenger market, with `market_served:` matching a market id from `custom_markets.yaml`, the renewal `parameters` (`life`, `limit`), and the ordered list of `subcategories` it draws from.

In [ ]:
print(Path("data/custom_fleet.yaml").read_text())

### 1.c — `custom_aircraft_inventory.yaml`

Two blocks:

* `reference_aircraft:` — the *old* and *recent* baseline aircraft each subcategory calibrates against (absolute values: energy per ASK, NOx and soot emission indices, base DOC, ASK/year per airframe, OEW, recurring/non-recurring costs).
* `aircraft:` — the new aircraft entering service. Each one references a `reference_aircraft` and a `entry_into_service_year`, and is described as **relative deltas** vs that reference (`consumption_evolution`, `nox_evolution`, `soot_evolution`, `doc_non_energy_evolution`, `hybridization_factor`).

In [ ]:
print(Path("data/custom_aircraft_inventory.yaml").read_text())

### 1.d — `config.yaml`

Glues the three input files together and selects the AeroMAPS model standards to run. The relevant blocks are `models.markets`, `models.fleet`, and `models.standards`.

In [ ]:
print(Path("data/config.yaml").read_text())

## 2. Build and run the process

Passing the config file to `create_process` is enough — the markets, fleet, and aircraft inventory are all picked up from the YAML references.

In [ ]:
from aeromaps import create_process

process = create_process(configuration_file="data/config.yaml")
process.compute()

## 3. Inspect the loaded markets and fleet

The `MarketManager` exposes every market declared in the YAML.

In [ ]:
for m in process.markets:
    print(f"{m.id:20s}  name={m.name!r:22s}  type={m.traffic_type:10s}  unit={m.traffic_unit}")

In [ ]:
for cat_name, cat in process.fleet_model.fleet.categories.items():
    subcats = [sc.name for sc in cat.subcategories.values()]
    print(f"[{cat.market_id:18s}] {cat_name!r:25s}  subcategories={subcats}")

## 4. Per-market results

Downstream outputs are now templated with the market id. The traffic series follow the convention `rpk_<market>` for passenger markets and `rtk_<market>` for freight markets; CO2 follows `co2_emissions_<market>`.

In [ ]:
vector = process.data["vector_outputs"]
years = [2019, 2030, 2050]

traffic_cols = ["rpk_regional", "rpk_intercontinental", "rtk_air_cargo"]
(vector[traffic_cols] / 1e9).loc[years].round(2)

In [ ]:
vector[["rpk", "ask", "load_factor", "co2_emissions_passenger", "co2_emissions_freight"]].loc[years]

### Per-market CO₂ emissions

In [ ]:
import matplotlib.pyplot as plt

market_ids = [m.id for m in process.markets]
co2_cols = [f"co2_emissions_{mid}" for mid in market_ids]

fig, ax = plt.subplots(figsize=(9, 4))
for col, mid in zip(co2_cols, market_ids):
    if col in vector.columns:
        ax.plot(vector.index, vector[col], label=mid)

ax.set_xlabel("Year")
ax.set_ylabel("CO₂ emissions (Mt)")
ax.set_title("CO₂ emissions per market")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 5. What to change to define your own scenario

To adapt this template to your case:

1. **Edit `custom_markets.yaml`** — add/remove market blocks, change `rpk_share_2019` / `rtk_share_2019` / `energy_share_2019` (keep the conservation constraints), and override any `defaults` block per market when needed (e.g. different growth curve, COVID profile, or hydrogen introduction year per market).
2. **Edit `custom_fleet.yaml`** — make sure every passenger market in `custom_markets.yaml` has a matching `market_served:` entry. Define the subcategories and their `share` (must sum to 100 % per market).
3. **Edit `custom_aircraft_inventory.yaml`** — add the reference and new aircraft referenced from the fleet.
4. **Leave `config.yaml` as-is** if you keep the same file names, otherwise update the three `*_data_file:` paths.

No Python code changes are needed: market names propagate through traffic, ASK, fleet, energy consumption, emissions, and cost models automatically.